In [1]:
import warnings
warnings.filterwarnings(
    "ignore",
    message="Possible clipped samples in output."
)

In [2]:
import os
import librosa
import numpy as np
import soundfile as sf
import pyloudnorm as pyln
from tqdm import tqdm

In [3]:
INPUT_ROOT = r"E:/Work/My Papers/Dravidian Lang Tech 2026/Dialect based speech processing/Dataset/Test"
OUTPUT_ROOT = r"E:/Work/My Papers/Dravidian Lang Tech 2026/Dialect based speech processing/Dataset/Test_processed_1"

TARGET_SR = 16000
TARGET_LUFS = -23.0

os.makedirs(OUTPUT_ROOT, exist_ok=True)

In [4]:
def load_audio(path):
    y, sr = librosa.load(path, sr=None, mono=False)
    return y, sr

In [5]:
def estimate_snr(y, eps=1e-9):
    signal_power = np.mean(y ** 2)
    noise_floor = np.percentile(np.abs(y), 10)
    noise_power = noise_floor ** 2
    return 10 * np.log10((signal_power + eps) / (noise_power + eps))

In [6]:
def select_best_channel(y):
    if y.ndim == 1:
        return y
    snrs = [estimate_snr(y[ch]) for ch in range(y.shape[0])]
    best_ch = int(np.argmax(snrs))
    return y[best_ch]

In [7]:
def resample_audio(y, sr):
    if sr != TARGET_SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR)
    return y, TARGET_SR

In [8]:
def remove_dc_offset(y):
    return y - np.mean(y)

In [9]:
def lufs_normalize(y, sr, target_lufs):
    meter = pyln.Meter(sr)
    loudness = meter.integrated_loudness(y)
    return pyln.normalize.loudness(y, loudness, target_lufs)

In [10]:
def safe_limiter(y, threshold=0.99):
    peak = np.max(np.abs(y))
    if peak > threshold:
        y = y * (threshold / peak)
    return y

In [11]:
# Collect all wav files first (for a clean tqdm bar)
wav_files = []
for root, _, files in os.walk(INPUT_ROOT):
    for file in files:
        if file.lower().endswith(".wav"):
            wav_files.append(os.path.join(root, file))

In [12]:
for input_path in tqdm(wav_files, desc="Preprocessing (Std + LUFS)", unit="file"):
    rel_path = os.path.relpath(os.path.dirname(input_path), INPUT_ROOT)
    output_dir = os.path.join(OUTPUT_ROOT, rel_path)
    os.makedirs(output_dir, exist_ok=True)

    output_path = os.path.join(output_dir, os.path.basename(input_path))

    # Load
    y, sr = load_audio(input_path)

    # Step 1 — Standardization
    y = select_best_channel(y)
    y, sr = resample_audio(y, sr)
    y = remove_dc_offset(y)

    # Step 2 — Loudness normalization (LUFS)
    try:
            y = lufs_normalize(y, sr, TARGET_LUFS)
            y = safe_limiter(y)
    except Exception:
        # Safety fallback for pathological short clips
        print('Safety Pass!')

    sf.write(output_path, y, sr)


Preprocessing (Std + LUFS):   0%|          | 0/579 [00:00<?, ?file/s]

Preprocessing (Std + LUFS): 100%|██████████| 579/579 [00:15<00:00, 37.20file/s]
